# Essai : le jeu corrigé (niveau 5) séparé en deux, contentieux et hors contentieux, puis résultats accumulés

**Question** : vaut-il mieux un seul modèle sur tous les clients, ou séparer le contentieux du reste ? Et une fois les résultats des deux populations **accumulés sur l'ensemble des clients**, combien de défauts récupère-t-on en tout, et avec quelle précision ? À comparer à l'étude d'origine (Yeh et Lien, 2009) et à la règle métier qui place **tout le contentieux en défaut**.

**Données** : `data/cleaned5_creditcard.csv` (niveau 5, le plus haut niveau de corrections), dont seules les **23 colonnes d'origine** sont des variables ; les colonnes du niveau 5 servent seulement à séparer les deux populations, par la règle retenue (définition 2) : client **au contentieux à M** si `FLAG_CTX` = 1 et `MOIS_SORTIE_CTX` = -1, **hors contentieux** sinon.

**Méthode, pour chaque population** : exactement celle de `essai_jeu_corrige.ipynb` (même code, repris tel quel) : trois variantes (complet, `PAY_n` seuls, sans les `PAY_n`), 4 modèles, boucle de GridSearch sur la précision moyenne avec un seuil de surapprentissage de 7 %, seuil de décision par la **règle marginale** (au plus 1 bon client signalé pour 1 défaut attrapé, à la marge, `TunedThresholdClassifierCV`), validation croisée stratifiée à 5 plis, `random_state=42`, importance par permutation par famille de colonnes.

**Synthèse globale, sur tous les clients du niveau 5** (section 6) : chaque client reçoit la décision de sa population (hors pli : prédit par un modèle qui ne l'a pas vu). Stratégies comparées, avec le **meilleur modèle de la variante complète** de chaque population (règle fixée avant : la plus haute précision moyenne hors pli, qui ne dépend d'aucun seuil) :
1. **règle seule** : tout le contentieux en défaut, personne d'autre ;
2. **règle + ML hors contentieux** : tout le contentieux en défaut, le modèle pour les autres clients ;
3. **règle + ML hors contentieux sans les `PAY_n`** : la même, avec un modèle qui ne voit que les montants (le comportement), pas les codifications ;
4. **deux ML** : un modèle pour le contentieux, un autre pour les autres clients ;
5. **un seul ML** sur tous les clients (`essai_jeu_corrige.ipynb`, lu dans le fichier qu'il enregistre, s'il a été lancé avec sa cellule d'enregistrement).

Pour chacune : défauts détectés (nombre et part de tous les défauts), clients signalés, précision, fausses alertes, taux d'erreur, et ratio d'aire (mesure de Yeh et Lien : qualité du tri, sans seuil).

⚠️ **Le test du ML est inclus** dans ces clients : aucune conclusion de cet essai ne sert à choisir le modèle final ; elle éclaire la problématique et la place de la règle du contentieux. **Ce n'est pas un scénario** : rien n'est écrit dans le fichier de résultats ni dans le journal, aucun modèle n'est enregistré ; les décisions, client par client, sont enregistrées dans `data/ML/comparatif_global/jeu_corrige_deux_populations.csv`.

**Étapes** :
1. chargement et séparation des deux populations ;
2. pipeline, critère et boucle (repris de `essai_jeu_corrige.ipynb`) ;
3. boucle d'optimisation, puis seuil optimisé, pour chaque population et chaque variante ;
4. ce qui porte les prédictions, dans chaque population (importance par famille) ;
5. comparaison unitaire : le modèle hors contentieux face aux scénarios (`ml_4`, `ml_14`), sur les mêmes clients ;
6. synthèse globale : résultats accumulés sur tous les clients, face à la règle seule, à un seul ML et à l'étude d'origine.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

COLONNES_ORIGINE = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + [f"PAY_{n}" for n in range(1, 7)] \
    + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
niveau_5 = pd.read_csv(root_dir / "data" / "cleaned5_creditcard.csv")
assert niveau_5["ID"].is_unique and not niveau_5[COLONNES_ORIGINE + ["dpnm"]].isna().any().any()
# Client au contentieux à M (définition 2, retenue) : FLAG_CTX = 1 et MOIS_SORTIE_CTX = -1 (pas de sortie)
au_ctx = ((niveau_5["FLAG_CTX"] == 1) & (niveau_5["MOIS_SORTIE_CTX"] == -1)).to_numpy()
POPULATIONS = {
    "Contentieux": niveau_5[au_ctx][["ID"] + COLONNES_ORIGINE + ["dpnm"]].reset_index(drop=True),
    "Hors contentieux": niveau_5[~au_ctx][["ID"] + COLONNES_ORIGINE + ["dpnm"]].reset_index(drop=True),
}
for nom, d in POPULATIONS.items():
    print(f"{nom} : {len(d)} clients, {d['dpnm'].sum()} défauts ({d['dpnm'].mean() * 100:.1f} %)")
print(f"Ensemble : {len(niveau_5)} clients, {niveau_5['dpnm'].sum()} défauts ({niveau_5['dpnm'].mean() * 100:.1f} %)")

# Population active : les cellules reprises de essai_jeu_corrige.ipynb travaillent sur « original » et « y_train » (ici, l'ensemble, pour les définitions)
original = niveau_5[["ID"] + COLONNES_ORIGINE + ["dpnm"]].reset_index(drop=True)
y_train = original["dpnm"].reset_index(drop=True)

## 2. Variantes, pipeline, critère et boucle : repris de `essai_jeu_corrige.ipynb`

In [ ]:
# ==============================================================================
# VARIABLES : les 23 variables d'origine, rangées selon leur encodage (celui des scénarios, méthode v8)
# ==============================================================================
PAY = [f"PAY_{n}" for n in range(1, 7)]

# PAY_n : nombres ordonnés (-2 à 8), mis à l'échelle (v6)
# PAY_AMTn : paiements très étirés, logarithme (v7)
# BILL_AMTn : factures, logarithme signé, car elles peuvent être négatives (v8)
# SEX, EDUCATION, MARRIAGE : catégories sans ordre, une colonne par catégorie (regroupées dès le niveau 0)
TYPES_ORIGINE = {
    "num": ["LIMIT_BAL", "AGE"] + PAY,
    "log": [f"PAY_AMT{n}" for n in range(1, 7)],
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"],
}

# Les trois variantes : colonnes gardées, chacune avec son encodage d'origine
VARIANTES = {
    "Complet": lambda c: True,
    "PAY_n seuls": lambda c: c in PAY,
    "Sans les PAY_n": lambda c: c not in PAY,
}
TYPES_VARIANTE = {v: {t: [c for c in cols if garder(c)] for t, cols in TYPES_ORIGINE.items()} for v, garder in VARIANTES.items()}
COLONNES_VARIANTE = {v: [c for cols in types.values() for c in cols] for v, types in TYPES_VARIANTE.items()}

# Contrôles : colonnes présentes, ni identifiant ni cible, les 23 variables couvertes par la variante complète
assert sorted(COLONNES_VARIANTE["Complet"]) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
assert sorted(COLONNES_VARIANTE["PAY_n seuls"] + COLONNES_VARIANTE["Sans les PAY_n"]) == sorted(COLONNES_VARIANTE["Complet"])
assert (original[TYPES_ORIGINE["log"]] >= 0).all().all(), "Le logarithme ne s'applique qu'à des montants positifs ou nuls"

display(pd.DataFrame({
    "Variante": list(VARIANTES),
    "Nombre de variables": [len(COLONNES_VARIANTE[v]) for v in VARIANTES],
    "Variables": [", ".join(COLONNES_VARIANTE[v]) for v in VARIANTES],
}).set_index("Variante"))

In [ ]:
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)   # textes affichés en entier

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    # Une branche par type de colonne, seulement pour les types présents dans la variante
    return ColumnTransformer(transformers=[(nom, TRANSFORMATIONS[nom](), colonnes)
                                           for nom, colonnes in TYPES_VARIANTE[variante].items() if colonnes])

# Critère optimisé par la boucle : la précision moyenne (aire sous la courbe précision-rappel), sur tous les rappels
CRITERE = "average_precision"
scorers = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}

# Rappel minimal des scénarios (60 %) : sert seulement à la comparaison avec les scénarios (section 6)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles de l'essai : les meilleurs des scénarios, avec leurs réglages de la première itération
MODELES_A_TESTER = ["LogisticRegression", "MLPClassifier", "RandomForest", "CatBoost"]

def modeles_de_base(variante):
    prep = preprocesseur(variante)
    tous = {
        "LogisticRegression": make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "MLPClassifier": make_pipeline(prep, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "RandomForest": make_pipeline(prep, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(prep, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

# ------------------------------------------------------------------------------
# Alertes et tableaux de scores (ceux des scénarios)
SEUIL_SURAPPRENTISSAGE = 0.07   # écart relatif train - val du critère : 7 % pour cet essai (5 % dans les scénarios) ; le double : surapprentissage fort
SEUIL_INSTABILITE = 0.02        # écart-type du critère sur les plis au-delà duquel on alerte (alerte seule, pas une contrainte)
METRIQUES = {"average_precision": "Précision moyenne", "roc_auc": "ROC AUC"}
LIBELLE_SCORE = "Précision des défauts prédits"
TAUX_DEFAUT = y_train.mean()    # précision, et précision moyenne, d'un tri au hasard

def apprend(score_val, std_val):
    # Plancher contre le sous-apprentissage : le critère doit dépasser le hasard (taux de défaut) de plus d'un écart-type
    return score_val - TAUX_DEFAUT > std_val

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, le critère, son écart-type et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Critère (val)"], ligne["Critère (train)"] = moy_val[CRITERE], moy_train[CRITERE]
    ligne["Écart-type critère (val)"] = std_val
    ligne["Écart train - val"] = moy_train[CRITERE] - moy_val[CRITERE]
    ligne["Écart relatif"] = ligne["Écart train - val"] / moy_val[CRITERE]
    ligne["Apprend"] = apprend(moy_val[CRITERE], std_val)
    ligne["Alerte"] = alerte(ligne["Écart relatif"]) if ligne["Apprend"] else "⚠️ N'APPREND RIEN (niveau du hasard)"
    if std_val > SEUIL_INSTABILITE:
        ligne["Alerte"] += " ; ⚠️ INSTABLE"
    return ligne

def pct(serie):
    return (serie * 100).round(1)

def parametres(params):
    # Paramètres sur une ligne : « depth 4, 5, 6 ; l2_leaf_reg 15 »
    return " ; ".join(f"{p} {', '.join(map(str, v)) if isinstance(v, list) else v}" for p, v in params.items())

def montrer(df, titre):
    display(Markdown(f"**{titre}**"))
    display(df.set_index(df.columns[0]))

In [ ]:
from itertools import product
from sklearn.model_selection import GridSearchCV

TOURS_SECURITE = 15   # garde-fou contre une boucle sans fin : la boucle s'arrête d'elle-même avant ; s'il est atteint, revoir les grilles ou les bornes

# Grilles du premier tour : celles des scénarios
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32, 16), (64, 32), (32,)], "mlpclassifier__alpha": [0.01, 0.1, 1.0]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10], "randomforestclassifier__min_samples_leaf": [20, 50, 100], "randomforestclassifier__n_estimators": [200, 300]},
    "CatBoost": {"catboostclassifier__depth": [4, 6], "catboostclassifier__l2_leaf_reg": [5, 10, 15], "catboostclassifier__learning_rate": [0.03, 0.05],
                 "catboostclassifier__iterations": [1000]},   # 1 000 arbres : valeur par défaut, réduite par la boucle en cas de surapprentissage
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre (ceux des scénarios) :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (architecture du MLP, nombre d'arbres du RandomForest) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
    "catboostclassifier__iterations":           {"simplifier": -1, "pas": ("x", 1.5), "bornes": (100, 3000),  "entier": True},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte, parmi les réglages qui battent le hasard : le meilleur critère sans surapprentissage ;
    # sinon le moins surappris ; si aucun ne bat le hasard, le meilleur score
    val, std = r[f"mean_test_{CRITERE}"], r[f"std_test_{CRITERE}"]
    ecarts = (r[f"mean_train_{CRITERE}"] - val) / val   # écarts relatifs
    utiles = apprend(val, std)
    sous_seuil = np.where(utiles & (ecarts <= SEUIL_SURAPPRENTISSAGE))[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(val[sous_seuil])])
    if utiles.any():
        idx = np.where(utiles)[0]
        return int(idx[np.argmin(ecarts[idx])])
    return int(np.argmax(val))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["bornes atteintes : la grille ne peut plus bouger"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille, X):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r[f"std_test_{CRITERE}"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def cle_reglage(params):
    # Identifiant d'une combinaison de paramètres, pour savoir si elle a déjà été testée
    return str(sorted(params.items()))

def reglages_nouveaux(grille, deja_testes):
    # Combinaisons de la grille qui n'ont encore été testées à aucun tour
    combinaisons = [dict(zip(grille, valeurs)) for valeurs in product(*grille.values())]
    return [c for c in combinaisons if cle_reglage(c) not in deja_testes]

def choisir_tour(tours):
    # Version retenue parmi les tours joués : d'abord ceux qui battent le hasard et sans surapprentissage
    # (sinon ceux qui battent le hasard, sinon tous) ; parmi eux, ceux à moins d'un écart-type du meilleur score
    # (à égalité avec lui) ; et parmi ces derniers, celui qui surapprend le moins
    for garder in (lambda l: l["Apprend"] and l["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE, lambda l: l["Apprend"], lambda l: True):
        groupe = [t for t in tours if garder(t[1])]
        if groupe:
            break
    meilleur = max(groupe, key=lambda t: t[1]["Critère (val)"])[1]
    egaux = [t for t in groupe if meilleur["Critère (val)"] - t[1]["Critère (val)"] <= meilleur["Écart-type critère (val)"]]
    return min(egaux, key=lambda t: t[1]["Écart relatif"])

def boucle(variante):
    # Boucle des scénarios, pour un jeu de variables : version retenue de chaque modèle, lignes de scores et historique
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    best_models, lignes_retenues, historique = {}, [], []
    for nom, modele in modeles_de_base(variante).items():
        grille, retenu, fin, deja_testes, reglage_precedent, tours = grilles[nom], None, None, set(), None, []
        for tour in range(1, TOURS_SECURITE + 1):
            grid, ligne = lancer_grille(nom, modele, grille, X)
            ligne["Tour"] = tour
            deja_testes |= {cle_reglage(c) for c in grid.cv_results_["params"]}
            surapprend = ligne["Écart relatif"] > SEUIL_SURAPPRENTISSAGE
            bords = bords_de_grille(grille, grid.best_params_)
            precedent = retenu
            tours.append((grid, ligne))
            retenu = choisir_tour(tours)
            grille_tour = {nom_court(p): v for p, v in grille.items()}
            reglage = ligne["Meilleurs paramètres"]
            # Stagnation : seulement entre deux tours sans surapprentissage
            stagne = (precedent is not None and not surapprend
                      and precedent[1]["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE
                      and ligne["Critère (val)"] - precedent[1]["Critère (val)"] <= precedent[1]["Écart-type critère (val)"])
            if not ligne["Apprend"]:
                fin = "sous-apprentissage : le réglage choisi ne bat plus le hasard"
            elif not surapprend and not bords:
                fin = "optimisé : ni surapprentissage ni paramètre au bord"
            elif reglage == reglage_precedent:
                fin = "convergé : le réglage choisi ne change plus"
            elif stagne:
                fin = "stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
            elif tour == TOURS_SECURITE:
                fin = "⚠️ garde-fou atteint : grilles ou bornes à revoir"
            else:
                grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
                if grille is None:
                    fin = actions[0]
                elif not reglages_nouveaux(grille, deja_testes):
                    fin = "convergé : la grille suivante ne contient aucun réglage nouveau"
            reglage_precedent = reglage
            historique.append({
                "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": reglage,
                "Critère (val)": ligne["Critère (val)"], "Écart-type": ligne["Écart-type critère (val)"],
                "Écart relatif": ligne["Écart relatif"], "Alerte": ligne["Alerte"],
                "Suite": fin if fin else "; ".join(actions),
            })
            if fin:
                break
        best_models[nom] = retenu[0].best_estimator_
        lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin, "Tours joués": tour}))
        print(f"  {nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")
    return best_models, pd.DataFrame(lignes_retenues), pd.DataFrame(historique)

## 3. Boucle d'optimisation et seuil optimisé, pour chaque population

Pour chaque population : la boucle règle les 4 modèles dans chaque variante (une ligne par modèle pendant le calcul), puis chaque version retenue est réentraînée sur chaque pli avec son **seuil par la règle marginale** (validation croisée interne), et jugée sur le pli de validation. Le tableau de chaque population donne, en validation (moyenne ± écart-type sur les 5 plis) : le réglage retenu, la précision moyenne et le ROC AUC (sans seuil), puis, au seuil optimisé, le rappel, la précision des défauts prédits, la précision marginale (contrôle de la règle, visée 50 %) et la part des clients signalés.

In [ ]:
import time
from sklearn.base import clone
from sklearn.metrics import average_precision_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV

cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # plis internes, pour le choix du seuil
BONS_CLIENTS_PAR_DEFAUT = 1   # règle fixée à l'avance par le métier : au plus 1 bon client signalé pour 1 défaut attrapé, à la marge
PART_MARGE = 0.10

def gain_net(y_vrai, y_pred):
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    return ((y_pred & (y_vrai == 1)).sum() - BONS_CLIENTS_PAR_DEFAUT * (y_pred & (y_vrai == 0)).sum()) / len(y_vrai)

SCORE_SEUIL = make_scorer(gain_net)

def precision_marginale(y_vrai, proba, signales):
    idx = np.where(signales)[0]
    if len(idx) == 0:
        return np.nan
    derniers = idx[np.argsort(proba[idx])[:max(1, int(np.ceil(PART_MARGE * len(idx))))]]
    return y_vrai[derniers].mean()

def moy_et(valeurs, en_pct=False):
    facteur = 100 if en_pct else 1
    return f"{np.nanmean(valeurs) * facteur:.{1 if en_pct else 3}f} ± {np.nanstd(valeurs) * facteur:.{1 if en_pct else 3}f}"

def activer(population):
    # Rend une population active pour les fonctions reprises de l'essai (elles lisent ces variables globales)
    global original, y_train, scale_pos, TAUX_DEFAUT, y, plis
    original = POPULATIONS[population]
    y_train = original["dpnm"].reset_index(drop=True)
    y = y_train.to_numpy()
    scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
    TAUX_DEFAUT = y_train.mean()
    plis = list(cv.split(original, y_train))

ETUDE = {}   # population -> {"probas", "decisions", "mesures", "modeles_plis", "plis", "retenus"}
for population in POPULATIONS:
    activer(population)
    debut = time.time()
    display(Markdown(f"### {population} : {len(original)} clients, taux de défaut {TAUX_DEFAUT * 100:.1f} %"))
    probas, decisions, mesures, modeles_plis, retenus = {}, {}, {}, {}, {}
    for variante in VARIANTES:
        print(f"Variante « {variante} »")
        best_models, df_retenus, _ = boucle(variante)
        retenus[variante] = df_retenus.set_index("Modèle")
        X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
        for nom, modele in best_models.items():
            p, d, entraines = np.zeros(len(y)), np.zeros(len(y), dtype=int), []
            m_plis = {"Rappel": [], "Précision": [], "Précision marginale": [], "Signalés": [], "Précision moyenne": [], "ROC AUC": []}
            for entrainement, validation in plis:
                seuil_auto = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
                seuil_auto.fit(X.iloc[entrainement], y[entrainement])
                p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
                d[validation] = seuil_auto.predict(X.iloc[validation])
                y_val, dec = y[validation], d[validation].astype(bool)
                m_plis["Rappel"].append(recall_score(y_val, dec))
                m_plis["Précision"].append(precision_score(y_val, dec, zero_division=0))
                m_plis["Précision marginale"].append(precision_marginale(y_val, p[validation], dec))
                m_plis["Signalés"].append(dec.mean())
                m_plis["Précision moyenne"].append(average_precision_score(y_val, p[validation]))
                m_plis["ROC AUC"].append(roc_auc_score(y_val, p[validation]))
                entraines.append(seuil_auto)
            probas[(variante, nom)], decisions[(variante, nom)], mesures[(variante, nom)], modeles_plis[(variante, nom)] = p, d, m_plis, entraines
    ETUDE[population] = {"probas": probas, "decisions": decisions, "mesures": mesures, "modeles_plis": modeles_plis, "plis": plis, "retenus": retenus}
    lignes = []
    for (variante, nom), m in mesures.items():
        r = retenus[variante].loc[nom]
        lignes.append({"Variante": variante, "Modèle": nom, "Réglage retenu": parametres(r["Meilleurs paramètres"]),
                       "Écart relatif (%)": round(r["Écart relatif"] * 100, 1),
                       "Précision moyenne": moy_et(m["Précision moyenne"]), "ROC AUC": moy_et(m["ROC AUC"]),
                       "Rappel (%)": moy_et(m["Rappel"], True), "Précision (%)": moy_et(m["Précision"], True),
                       "Précision marginale (%)": moy_et(m["Précision marginale"], True), "Clients signalés (%)": moy_et(m["Signalés"], True)})
    display(Markdown(f"**{population} : détection en validation, au seuil de la règle marginale** ({(time.time() - debut) / 60:.1f} min)"))
    display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

**Comment lire** : chaque population a son propre taux de défaut, qui est aussi la précision d'un tri au hasard. Au contentieux, où le défaut est la règle, la règle marginale signale presque tout le monde : la question y est plutôt de reconnaître ceux qui **ne feront pas** défaut. Hors contentieux, elle ne signale que la tête de liste. La précision marginale vérifie la règle (proche de 50 %).

## 4. Ce qui porte les prédictions, dans chaque population

Importance par permutation (perte de précision moyenne quand une famille de colonnes est mélangée, sur les plis de validation), comme dans `essai_jeu_corrige.ipynb` : part de chaque famille dans la perte totale, par population, variante et modèle.

In [ ]:
FAMILLES = {
    "Codifications (PAY_n)": PAY,
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
REPETITIONS = 5
generateur = np.random.default_rng(42)

def perte_par_permutation(modeles, X, colonnes):
    # Perte de précision moyenne sur chaque pli de validation quand les colonnes sont mélangées d'un bloc (même tirage pour toutes),
    # moyennée sur les répétitions
    pertes = []
    for m, (_, validation) in zip(modeles, plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        perte = []
        for _ in range(REPETITIONS):
            melange = X_val.copy()
            ordre_melange = generateur.permutation(len(melange))
            melange[colonnes] = melange[colonnes].to_numpy()[ordre_melange]
            perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
        pertes.append(np.mean(perte))
    return pertes


for population, e in ETUDE.items():
    activer(population)
    pertes = {}
    for (variante, nom), modeles in e["modeles_plis"].items():
        X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
        for famille, colonnes in FAMILLES.items():
            presentes = [c for c in colonnes if c in X.columns]
            if presentes:
                pertes[(variante, nom, famille)] = np.mean(perte_par_permutation(modeles, X, presentes))
    cles = list(dict.fromkeys((v, n) for v, n, _ in pertes))
    tableau = pd.DataFrame({cle: {f: pertes[(*cle, f)] for f in FAMILLES if (*cle, f) in pertes} for cle in cles})
    part = tableau.clip(lower=0) / tableau.clip(lower=0).sum() * 100
    display(Markdown(f"**{population} : part de chaque famille dans la perte de précision moyenne (%)**"))
    display(part.round(1).rename_axis(index="Famille", columns=["Variante", "Modèle"]))

## 5. Comparaison unitaire : le modèle hors contentieux face aux scénarios, sur les mêmes clients

Seule comparaison directe avec le travail mené dans les scénarios : les probabilités hors pli du modèle hors contentieux, restreintes aux **clients du train ML** et découpées selon **les 5 plis des scénarios**, face aux probabilités hors pli enregistrées par `ml_4` (référence des scénarios) et `ml_14` (version finale). Mesure des scénarios : le **score décisionnel F2 au rappel minimal de 60 %**, pli par pli ; même verdict. Les deux côtés n'ont pas de contentieux ; le modèle de l'essai a cependant appris aussi sur les clients du test du ML et sur ceux hors du périmètre du ML, avec les 23 colonnes d'origine seulement, quand les scénarios ajoutent des colonnes construites. Le contentieux n'a pas d'équivalent dans les scénarios : sa comparaison unitaire se lit avec la règle, en section 6.

In [ ]:
from sklearn.metrics import fbeta_score

def f2_au_rappel_minimal(y_vrai, proba):
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

activer("Hors contentieux")
probas = ETUDE["Hors contentieux"]["probas"]
# ==============================================================================
# SCÉNARIOS COMPARÉS : numéros des scénarios dont les probabilités hors pli (méthode v8) sont lues
# ==============================================================================
SCENARIOS_COMPARES = ["4", "14"]   # ml_4 : référence des scénarios ; ml_14 : version finale
VERSION_METHODE = "v8"

dossier_probas = root_dir / "data" / "ML" / "probas_hors_pli"
train_ml = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv", usecols=["ID", "dpnm"])
y_ml = train_ml["dpnm"].to_numpy()
plis_ml = list(cv.split(train_ml, train_ml["dpnm"]))   # les 5 plis des scénarios (même ordre des clients, même graine)

# Probabilités hors pli du jeu hors contentieux, restreintes aux clients du train ML, dans l'ordre du train ML
position = pd.Series(np.arange(len(original)), index=original["ID"])
assert train_ml["ID"].isin(position.index).all(), "Des clients du train ML manquent dans le jeu hors contentieux"
lignes_ml = position.loc[train_ml["ID"]].to_numpy()
assert (y[lignes_ml] == y_ml).all(), "Cible différente entre le jeu hors contentieux et le train ML"

f2_ml = {}   # (source, modèle) -> F2 sur les 5 plis des scénarios
for (variante, nom), p in probas.items():
    p_ml = p[lignes_ml]
    f2_ml[(f"Hors contentieux, {variante}", nom)] = [f2_au_rappel_minimal(y_ml[val], p_ml[val]) for _, val in plis_ml]

for numero in SCENARIOS_COMPARES:
    ref = pd.read_csv(dossier_probas / f"ml_{numero}_{VERSION_METHODE}.csv").set_index("ID").loc[train_ml["ID"]]
    assert (ref["dpnm"].to_numpy() == y_ml).all(), f"Probabilités de ml_{numero} non alignées sur le train ML"
    for nom in [m for m in MODELES_A_TESTER if m in ref.columns]:
        f2_ml[(f"ml_{numero}", nom)] = [f2_au_rappel_minimal(y_ml[val], ref[nom].to_numpy()[val]) for _, val in plis_ml]

sources = list(dict.fromkeys(source for source, _ in f2_ml))
tableau = pd.DataFrame({source: {nom: f"{np.mean(f2_ml[(source, nom)]):.3f} ± {np.std(f2_ml[(source, nom)]):.3f}"
                                 for nom in MODELES_A_TESTER if (source, nom) in f2_ml} for source in sources}).T
display(Markdown(f"**Lecture 1 : sur les {len(train_ml)} clients du train ML, plis des scénarios** (score décisionnel F2 ± écart-type sur les plis ; "
                 f"taux de défaut {y_ml.mean() * 100:.1f} %)"))
display(tableau.rename_axis("Source"))

In [ ]:
def meilleur_de(source):
    return max([nom for s, nom in f2_ml if s == source], key=lambda nom: np.mean(f2_ml[(source, nom)]))

lignes = []
for variante in VARIANTES:
    source = f"Hors contentieux, {variante}"
    for numero in SCENARIOS_COMPARES:
        scenario = f"ml_{numero}"
        paires = [(nom, nom) for nom in MODELES_A_TESTER if (scenario, nom) in f2_ml] + [(meilleur_de(scenario), meilleur_de(source))]
        for i, (nom_ref, nom_base) in enumerate(paires):
            gains = np.array(f2_ml[(source, nom_base)]) - np.array(f2_ml[(scenario, nom_ref)])
            lignes.append({
                "Hors contentieux": variante,
                "Face à": scenario,
                "Modèle": nom_ref if i < len(paires) - 1 else f"Meilleur contre meilleur ({nom_ref} → {nom_base})",
                "F2 scénario": round(np.mean(f2_ml[(scenario, nom_ref)]), 3),
                "F2 jeu hors contentieux": round(np.mean(f2_ml[(source, nom_base)]), 3),
                "Gain du jeu hors contentieux": f"{gains.mean():+.3f} ± {gains.std():.3f}",
                "Plis gagnés": f"{(gains > 0).sum()} sur 5",
                "Verdict": verdict(gains),
            })
display(Markdown("**Lecture 1 : le jeu hors contentieux face aux scénarios, pli par pli, sur les mêmes clients**"))
display(pd.DataFrame(lignes).set_index(["Hors contentieux", "Face à", "Modèle"]))

## 6. Synthèse globale : résultats accumulés sur tous les clients du niveau 5

Chaque client reçoit la décision (et la probabilité) de sa population, hors pli. Le **meilleur modèle** de chaque population est celui de la variante choisie qui a la plus haute précision moyenne hors pli (règle fixée avant de lancer). Les mesures portent sur **tous les clients** : défauts détectés (nombre et part de tous les défauts), clients signalés, précision des défauts prédits, fausses alertes, défauts manqués, taux d'erreur (fausses alertes et défauts manqués, sur tous les clients), et ratio d'aire (Yeh et Lien : aire entre la courbe de gain et la diagonale, rapportée à celle de la courbe parfaite ; il vaut 2 × ROC AUC − 1).

Pour le ratio d'aire, les clients sont triés par une note : la règle place le contentieux en tête ; « deux ML » trie tous les clients par la probabilité de leur modèle (deux modèles différents : probabilités pas tout à fait comparables, ratio indicatif).

In [ ]:
from sklearn.metrics import roc_auc_score

def meilleur_modele(population, variante):
    e = ETUDE[population]
    return max([n for v, n in e["probas"] if v == variante], key=lambda n: average_precision_score(POPULATIONS[population]["dpnm"], e["probas"][(variante, n)]))

def vecteurs(population, variante):
    # Probabilités et décisions hors pli du meilleur modèle d'une population, rangées par identifiant client
    nom = meilleur_modele(population, variante)
    ids = POPULATIONS[population]["ID"].to_numpy()
    return nom, pd.Series(ETUDE[population]["probas"][(variante, nom)], index=ids), pd.Series(ETUDE[population]["decisions"][(variante, nom)], index=ids)

ids_tous = niveau_5["ID"].to_numpy()
y_tous = niveau_5["dpnm"].to_numpy()
ctx = pd.Series(au_ctx, index=ids_tous)

nom_ctx, p_ctx, d_ctx = vecteurs("Contentieux", "Complet")
nom_hors, p_hors, d_hors = vecteurs("Hors contentieux", "Complet")
nom_hors_sp, p_hors_sp, d_hors_sp = vecteurs("Hors contentieux", "Sans les PAY_n")

def assembler(dec_ctx, dec_hors, note_ctx, note_hors):
    dec = pd.concat([dec_ctx, dec_hors]).reindex(ids_tous).to_numpy()
    note = pd.concat([note_ctx, note_hors]).reindex(ids_tous).to_numpy()
    return dec.astype(int), note

ids_ctx, ids_hors = POPULATIONS["Contentieux"]["ID"], POPULATIONS["Hors contentieux"]["ID"]
un = lambda ids: pd.Series(1, index=ids)
zero = lambda ids: pd.Series(0, index=ids)
STRATEGIES = {
    "1. Règle seule (contentieux en défaut)": assembler(un(ids_ctx), zero(ids_hors), un(ids_ctx) * 2.0, zero(ids_hors) * 1.0),
    f"2. Règle + ML hors contentieux ({nom_hors})": assembler(un(ids_ctx), d_hors, un(ids_ctx) * 2.0, p_hors),
    f"3. Règle + ML hors contentieux sans les PAY_n ({nom_hors_sp})": assembler(un(ids_ctx), d_hors_sp, un(ids_ctx) * 2.0, p_hors_sp),
    f"4. Deux ML ({nom_ctx} au contentieux, {nom_hors} hors contentieux)": assembler(d_ctx, d_hors, p_ctx, p_hors),
}
# 5. Un seul ML sur tous les clients : décisions enregistrées par essai_jeu_corrige.ipynb (cellule d'enregistrement)
fichier_un_ml = root_dir / "data" / "ML" / "comparatif_global" / "jeu_corrige.csv"
if fichier_un_ml.exists():
    un_ml = pd.read_csv(fichier_un_ml).set_index("ID").loc[ids_tous]
    noms = sorted({c.split(" | ")[2] for c in un_ml.columns if c.startswith("proba | Complet |")})
    nom_un = max(noms, key=lambda n: average_precision_score(y_tous, un_ml[f"proba | Complet | {n}"]))
    STRATEGIES[f"5. Un seul ML sur tous les clients ({nom_un})"] = (un_ml[f"decision | Complet | {nom_un}"].to_numpy().astype(int), un_ml[f"proba | Complet | {nom_un}"].to_numpy())
else:
    display(Markdown("⚠️ Stratégie 5 absente : lancer `essai_jeu_corrige.ipynb` avec sa cellule d'enregistrement (fin de la section 4)."))

def ratio_aire(y_vrai, note):
    n, d = len(y_vrai), y_vrai.sum()
    cumul = np.cumsum(y_vrai[np.argsort(-note, kind="mergesort")])
    diagonale = d * (n + 1) / 2
    return (cumul.sum() - diagonale) / (np.minimum(np.arange(1, n + 1), d).sum() - diagonale)

lignes = []
for nom, (dec, note) in STRATEGIES.items():
    signale = dec == 1
    detectes = (signale & (y_tous == 1)).sum()
    fausses = (signale & (y_tous == 0)).sum()
    manques = (~signale & (y_tous == 1)).sum()
    lignes.append({"Stratégie": nom, "Défauts détectés": detectes, "Part des défauts (%)": round(detectes / y_tous.sum() * 100, 1),
                   "Clients signalés": signale.sum(), "Part des clients signalés (%)": round(signale.mean() * 100, 1),
                   "Précision des défauts prédits (%)": round(detectes / signale.sum() * 100, 1) if signale.sum() else np.nan,
                   "Fausses alertes": fausses, "Défauts manqués": manques,
                   "Taux d'erreur": round((fausses + manques) / len(y_tous), 3), "Ratio d'aire": round(ratio_aire(y_tous, note), 3)})
lignes.append({"Stratégie": "Personne n'est signalé (repère)", "Défauts détectés": 0, "Part des défauts (%)": 0.0, "Clients signalés": 0,
               "Part des clients signalés (%)": 0.0, "Précision des défauts prédits (%)": np.nan, "Fausses alertes": 0,
               "Défauts manqués": y_tous.sum(), "Taux d'erreur": round(y_tous.mean(), 3), "Ratio d'aire": 0.0})
display(Markdown(f"**Résultats accumulés sur tous les clients du niveau 5** ({len(y_tous)} clients, {y_tous.sum()} défauts, taux de défaut {y_tous.mean() * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index("Stratégie"))

# Détail par population : d'où viennent les défauts détectés et les fausses alertes
detail = []
for nom, (dec, _) in STRATEGIES.items():
    for population, masque in (("Contentieux", au_ctx), ("Hors contentieux", ~au_ctx)):
        s, yy = dec[masque] == 1, y_tous[masque]
        detail.append({"Stratégie": nom, "Population": population, "Défauts détectés": (s & (yy == 1)).sum(), "sur": yy.sum(),
                       "Fausses alertes": (s & (yy == 0)).sum(), "Précision (%)": round((s & (yy == 1)).sum() / s.sum() * 100, 1) if s.sum() else np.nan})
display(Markdown("**Détail par population**"))
display(pd.DataFrame(detail).set_index(["Stratégie", "Population"]))

# Comparatif avec l'étude d'origine : ses 6 méthodes (Tableau 1, en validation) et nos stratégies globales, sur les deux mesures qu'elle publie
YEH_LIEN = {   # méthode : (taux d'erreur, ratio d'aire), en validation
    "Réseau de neurones": (0.17, 0.54), "Arbre de classification": (0.17, 0.536), "Bayésien naïf": (0.21, 0.53),
    "KNN": (0.16, 0.45), "Régression logistique": (0.18, 0.44), "Analyse discriminante": (0.26, 0.43),
}
comparatif = [{"Source": "Yeh et Lien (2009)", "Méthode ou stratégie": m, "Taux d'erreur": e, "Ratio d'aire": a,
               "Périmètre": "fichier brut, tous les clients", "Découpage": "un seul (entraînement / validation)"} for m, (e, a) in YEH_LIEN.items()]
comparatif += [{"Source": "Cet essai", "Méthode ou stratégie": l["Stratégie"], "Taux d'erreur": l["Taux d'erreur"], "Ratio d'aire": l["Ratio d'aire"],
                "Périmètre": f"niveau 5 ({len(y_tous)} clients)", "Découpage": "validation croisée à 5 plis (décisions hors pli)"} for l in lignes]
display(Markdown("**Comparatif avec l'étude d'origine (Yeh et Lien, 2009, Tableau 1, en validation)** : taux d'erreur et ratio d'aire, "
                 "les deux seules mesures qu'elle publie (ni rappel ni précision)"))
display(pd.DataFrame(comparatif).set_index(["Source", "Méthode ou stratégie"]))
display(Markdown(
    "**Périmètres différents, et pourquoi c'est défendable** : l'étude d'origine travaille sur le fichier brut ; cet essai, sur le niveau 5 du nettoyage, "
    "qui retire les paiements géants (montants hors de toute proportion avec le plafond), les comptes inactifs (aucune facture positive sur les six mois "
    "et aucun paiement de mai à septembre : rien à prédire sur leur comportement de paiement) et les plafonds atypiques (au-delà de 500 000 NT$, "
    "une clientèle à part), et corrige la codification des `PAY_n`. Ces retraits sont justifiés par le métier au nettoyage, jamais par leur effet sur un score. "
    "La comparaison reste **indicative** : clients, découpage et taux de défaut diffèrent, et les chiffres de l'étude sont arrondis au centième. "
    "Elle montre où se situe chaque stratégie par rapport à la référence publiée sur ces données."))

# Décisions client par client, pour l'accumulation ultérieure
sortie = pd.DataFrame({"ID": ids_tous, "dpnm": y_tous, "contentieux": au_ctx.astype(int)})
for nom, (dec, note) in STRATEGIES.items():
    sortie[f"decision | {nom}"] = dec
    sortie[f"note | {nom}"] = note
dossier_sortie = root_dir / "data" / "ML" / "comparatif_global"
dossier_sortie.mkdir(parents=True, exist_ok=True)
sortie.to_csv(dossier_sortie / "jeu_corrige_deux_populations.csv", index=False)
print(f"Décisions enregistrées : {dossier_sortie / 'jeu_corrige_deux_populations.csv'}")

**Comment lire** :
- **Défauts détectés et précision** : combien de défauts chaque stratégie récupère en tout, et à quel prix (fausses alertes). La règle seule donne le repère : les défauts du contentieux, quasi certains, sans modèle.
- **Règle + ML** (2 et 3) : ce que le modèle ajoute aux défauts du contentieux, en ne signalant hors contentieux que la tête de liste (règle marginale). La stratégie 3 montre ce que le **comportement seul** (les montants, sans codification) ajoute à la règle.
- **Deux ML et un seul ML** (4 et 5) : si un modèle fait aussi bien que la règle sur le contentieux, et si séparer les populations change quelque chose.
- **Comparatif avec l'étude d'origine** : nos stratégies face à ses 6 méthodes, sur ses deux mesures (taux d'erreur, ratio d'aire) ; indicatif (périmètre, découpage et taux de défaut différents), mais défendable : les clients retirés le sont pour des raisons métier. Le taux d'erreur ne dit pas combien de défauts sont trouvés : « personne n'est signalé » en donne le repère.